In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

output_folder = "outputs"

coins = ["BTC", "ETH", "BNB", "ADA", "SOL", "DOGE"]
row_order = ["Bubble Creation", "Not Bubble", "Bubble Collapse"]
col_order = ["Extreme Fear", "Fear", "Neutral", "Greed", "Extreme Greed"]

In [31]:
def merge_data():
    bubble_labels = pd.read_csv("bubble_labels.csv").rename(columns = {"timestamp": "date"})
    int_data = pd.read_csv("integrated_data.csv")
    fg = int_data[["date", "value_classification"]].drop_duplicates("date")
    return bubble_labels.merge(fg, on = "date", how = "inner")

def build_matrices(df, coin, config):
    sub = df[df["symbol"] == coin]
    ct = pd.crosstab(sub[f"gsadf_label_{config}"], sub["value_classification"])
    ct = ct.reindex(index = row_order, columns = col_order, fill_value = 0)
    row_sums = ct.sum(axis = 1).replace(0, np.nan)
    return (ct.div(row_sums, axis = 0) * 100).fillna(0)

In [32]:
def plot_coins(ax, matrix, coin):
    ax.axis("off")
    ax.text(0, 1.0, coin, fontsize = 11, fontweight = "bold", va = "bottom")
    col_x = [0.34 + i * 0.135 for i in range(len(col_order))]
    
    for x, label in zip(col_x, col_order):
        ax.text(x, 0.88, label, ha = "center", va = "bottom", fontsize = 8, color = "#555")
    ax.axhline(0.86, color = "#ccc", linewidth = 0.8)

    for y, row in zip([0.7, 0.5, 0.3], row_order):
        ax.text(0, y, row, va = "center", ha = "left", fontsize = 8)
        for x, col in zip(col_x, col_order):
            ax.text(x, y, f"{matrix.loc[row, col]:.1f}%", ha = "center", va = "center", fontsize = 8)

    ax.axhline(0.16, color = "#ccc", linewidth = 0.5)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)

def make_fig(merged, config):
    fig, axes = plt.subplots(2, 3, figsize = (16, 6))
    fig.suptitle(f"GSADF Bubble Labels x Fear & Greed Index ({config}% CI)", fontsize = 13, fontweight = "bold", y = 1.05)
    for ax, coin in zip(axes.flat, coins):
        plot_coins(ax, build_matrices(merged, coin, config), coin)
    fig.tight_layout()
    return fig

if __name__ == "__main__":
    os.makedirs(output_folder, exist_ok = True)
    merged = merge_data()
    for config in ["95", "90"]:
        fig = make_fig(merged, config)
        path = os.path.join(output_folder, f"co-occurence_{config}ci.png")
        fig.savefig(path, dpi = 150, bbox_inches = "tight")
        plt.close(fig)
        
    